# 03. curves: 레시피 + 시세 → QuantLib 커브, 그리고 흔들 수 있는 커브

`src/quanttool/market/curves.py`

| 의존하는 것 | 제공하는 것 |
|---|---|
| `config` (관례·커브 레시피·지수), 시세 하루치 | `HELPERS`, `build_curve` → (커브, 시세 Quote), `CurveFactors`(시장 객체) |

**객체 지도** (이 순서로 본다)

1. 날짜 도구: `to_ql_date`, `evaluation_date`, `bill_price`
2. `Context`: 헬퍼 함수가 받는 이 커브 공통 재료
3. `HELPERS`: 구간 종류 → 헬퍼 함수 (5종 하나씩)
4. `build_curve`: 얼림 vs linked, QuantLib 직접 조립과 비교, 듀얼 커브
5. `CurveFactors`: 기준 커브 + 격자 이동량 층, `roll`

> 데이터는 `tests/fixtures/`의 **합성 샘플**이다. 구조를 보는 용도이고 숫자로 시장을 해석하지 않는다.

In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd
import QuantLib as ql

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
FIX = ROOT / "tests" / "fixtures"  # 합성 샘플 시세·예시 북
ASOF = date(2026, 10, 23)

from quanttool.config import load_config
from quanttool.market.quotes import read_quotes

cfg = load_config(ROOT / "configs")
quotes = read_quotes(FIX / "market_sample.txt")
q = quotes[ASOF]

## 날짜 도구
`evaluation_date(d)`는 QuantLib 전역 평가일을 잠시 d로 두고 블록이 끝나면 되돌린다. `bill_price`는 재무부 bill의 BEY를 액면 100 가격으로 바꾼다(QuantLib에 없음).

In [ ]:
from quanttool.market.curves import bill_price, evaluation_date, to_ql_date

S = ql.Settings.instance()
print("to_ql_date:", to_ql_date(ASOF))
before = S.evaluationDate
with evaluation_date(ASOF):
    print("블록 안:", S.evaluationDate)
print("블록 밖 되돌림:", S.evaluationDate == before)
print("bill BEY 3.95%, 91일 → 가격", round(bill_price(0.0395, 91), 6))

## `Context`: 헬퍼 함수가 받는 이 커브 공통 재료
한 커브의 헬퍼들은 관례·달력·결제일·할인 커브·지수 만드는 법이 모두 같다. `build_curve`가 커브마다 한 번 묶어 헬퍼 함수 모두에 넘긴다. 그래서 헬퍼 함수 모양이 `(Quote, 만기, 구간, ctx)`로 같고, 종류만 바꿔 부르는 `HELPERS` 사전이 된다.

In [ ]:
from dataclasses import fields

from quanttool.market.curves import Context

[f.name for f in fields(Context)]

## `HELPERS`: 구간 종류 → 헬퍼 함수
헬퍼 = 부트스트랩이 맞춰야 할 조건 하나(이 상품이 이 시세로 거래된다). 시세 하나에 헬퍼 하나. 부트스트랩 방법은 `PiecewiseXxx` 하나뿐이고, 구간 종류마다 다른 것은 "시세를 어떤 상품 조건으로 바꾸나"다.

In [ ]:
from quanttool.market.curves import HELPERS


def ctx_for(curve):
    cc, conv = cfg.curves[curve], cfg.conventions[cfg.curves[curve].conventions]
    cal, today = conv.ql_calendar(), to_ql_date(ASOF)
    return cc, Context(
        curve,
        conv,
        cal,
        cal.advance(today, conv.settlement_days, ql.Days),
        ql.YieldTermStructureHandle(),
        lambda n: cfg.indices[n].make(n),
    )


rows, helpers = [], []
with evaluation_date(ASOF):
    cc, ctx = ctx_for("UST")  # UST 커브의 헬퍼 13개를 손으로
    for seg in cc.segments:
        for tenor in seg.tenors:
            key = cc.key(seg, tenor)
            h = HELPERS[seg.type](ql.SimpleQuote(q[key]), tenor, seg, ctx)
            helpers.append(h)
            rows.append(
                {
                    "구간 종류": seg.type,
                    "key": key,
                    "시세": q[key],
                    "QuantLib 헬퍼": type(h).__name__,
                    "만기": h.maturityDate().ISO(),
                    "헬퍼가 맞추는 값": h.quote().value(),
                }
            )
    curve = ql.PiecewiseLogCubicDiscount(
        to_ql_date(ASOF), helpers, ql.Actual365Fixed()
    )  # 부트스트랩: 하나가 전부를 맞춘다
    curve.discount(1.0)
    for r, h in zip(rows, helpers, strict=True):
        r["커브로 다시 계산"] = h.impliedQuote()
pd.DataFrame(rows)

Bill·Bond 헬퍼는 **가격**(할인채 가격, par 채권 100)을, OIS·Deposit·Swap 헬퍼는 **금리**를 맞춘다. "커브로 다시 계산"이 "맞추는 값"과 같으면 부트스트랩이 시세를 재현한 것이다. OIS(KOFR)·Deposit·Swap(CD91) 헬퍼는 아래 `build_curve`와 듀얼 커브에서 같은 방식으로 만들어진다.

In [ ]:
from quanttool.market.curves import build_curve

ust, _ = build_curve(cfg.curves["UST"], cfg.conventions["UST"], q, ASOF)
kofr, _ = build_curve(cfg.curves["KOFR"], cfg.conventions["KRW_OIS"], q, ASOF, indices=cfg.indices)
with evaluation_date(ASOF):
    print("손으로 = build_curve (10Y DF 차이):", curve.discount(10.0) - ust.discount(10.0))

## `build_curve(레시피, 관례, 시세, asof, linked, curves, indices)` → `(커브, {시세 키: SimpleQuote})`

In [ ]:
curve, used = build_curve(cfg.curves["UST"], cfg.conventions["UST"], q, ASOF)
print(type(curve).__name__, "기준일", curve.referenceDate(), "|", len(used), "시세")
{k: v.value() for k, v in list(used.items())[:4]}

QuantLib으로 직접 조립한 커브와 같다 (bill: BEY → 할인채 가격, 쿠폰채: par yield = 쿠폰인 가격 100 채권).

In [ ]:
with evaluation_date(ASOF):
    today = to_ql_date(ASOF)
    cal = ql.UnitedStates(ql.UnitedStates.GovernmentBond)
    settle = cal.advance(today, 1, ql.Days)
    hs = []
    for t in ["1M", "2M", "3M", "4M", "6M", "1Y"]:
        mat = cal.adjust(settle + ql.Period(t), ql.Following)
        zcb = ql.ZeroCouponBond(1, cal, 100.0, mat, ql.Following, 100.0, settle)
        price = bill_price(q[f"BOND/YIELD/USD/UST/PAR/{t}"], mat - settle)
        hs.append(ql.BondHelper(ql.QuoteHandle(ql.SimpleQuote(price)), zcb))
    for t in ["2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]:
        sch = ql.Schedule(
            settle,
            settle + ql.Period(t),
            ql.Period(ql.Semiannual),
            cal,
            ql.Unadjusted,
            ql.Unadjusted,
            ql.DateGeneration.Backward,
            False,
        )
        hs.append(
            ql.FixedRateBondHelper(
                ql.QuoteHandle(ql.SimpleQuote(100.0)),
                1,
                100.0,
                sch,
                [q[f"BOND/YIELD/USD/UST/PAR/{t}"]],
                ql.ActualActual(ql.ActualActual.Bond),
                ql.Unadjusted,
                100.0,
                settle,
            )
        )
    direct = ql.PiecewiseLogCubicDiscount(today, hs, ql.Actual365Fixed())
    print("10Y 할인계수 차이:", curve.discount(10.0) - direct.discount(10.0))

**얼림 vs linked**: 기본은 부트스트랩 직후 얼려 시세가 바뀌어도 그대로다. `linked=True`면 돌려받은 시세 Quote를 바꾸면 커브가 다시 풀린다.

In [ ]:
for linked in [False, True]:
    c, used = build_curve(cfg.curves["UST"], cfg.conventions["UST"], q, ASOF, linked=linked)
    s = used["BOND/YIELD/USD/UST/PAR/10Y"]
    with evaluation_date(ASOF):
        before = c.zeroRate(10.0, ql.Continuous).rate()
        s.setValue(s.value() + 0.0010)
        print(
            f"linked={linked}: 10Y 시세 +10bp → 10Y 제로 {(c.zeroRate(10.0, ql.Continuous).rate() - before) * 1e4:.2f}bp"
        )

**듀얼 커브**: CD91의 스왑 헬퍼는 `curves`로 받은 KOFR Handle로 할인한다(`discount_curve: KOFR`). 그래서 KOFR를 먼저 만든다.

In [ ]:
handles = {"KOFR": ql.YieldTermStructureHandle(kofr)}
cd91, cd91_quotes = build_curve(
    cfg.curves["CD91"], cfg.conventions["KRW_IRS"], q, ASOF, curves=handles, indices=cfg.indices
)
with evaluation_date(ASOF):
    print(
        {
            n: round(c.zeroRate(5.0, ql.Continuous).rate() * 100, 4)
            for n, c in [("KOFR", kofr), ("CD91", cd91)]
        },
        "(5Y 제로 %)",
    )
list(cd91_quotes)

## `CurveFactors`: 기준 커브 + 격자 이동량 층 (시장 객체)
```
handle (상품이 연결) → PiecewiseZeroSpreadedTermStructure(_base_link, 격자 이동량 Quote들)
                                                  └ _base_link → 기준 커브 (roll이 갈아 끼움)
```
이동량이 0이면 기준 커브와 정확히 같다. 격자점 사이는 선형, 바깥은 평탄. `.base`가 얼린 원래 커브다.

In [ ]:
from quanttool.market.curves import CurveFactors

cf = CurveFactors(ust, ASOF, cfg.curves["UST"].factor_tenors)
print("tenors", cf.tenors[:6], "…")
print("times", {k: round(v, 4) for k, v in list(cf.times.items())[:4]}, "…")
print(
    "factors",
    type(cf.factors()["5Y"]).__name__,
    cf.factors()["5Y"].value(),
    "| handle",
    type(cf.handle).__name__,
)
with evaluation_date(ASOF):
    print("기준 상태 차이:", cf.handle.discount(7.3) - cf.base.discount(7.3))

5Y 이동량 하나를 1bp로: 5Y에서 1bp, 4.5Y(4Y·5Y 가운데)에서 0.5bp, 이웃 격자점 4Y·6Y에서 0.

In [ ]:
def zero_bp(c, t):
    return c.zeroRate(t, ql.Continuous).rate() * 1e4


t = cf.times
cf.factors()["5Y"].setValue(1e-4)
with evaluation_date(ASOF):
    at = {"4Y": t["4Y"], "4.5Y": (t["4Y"] + t["5Y"]) / 2, "5Y": t["5Y"], "6Y": t["6Y"]}
    moved = pd.Series(
        {k: zero_bp(cf.handle, x) - zero_bp(cf.base, x) for k, x in at.items()}, name="제로 이동 bp"
    )
cf.factors()["5Y"].setValue(0.0)
moved.round(4)

`roll(d)`: 기준 커브를 `ImpliedTermStructure(오늘 커브, d)`로 갈아 끼운다 (ForwardForward: DF_d(t) = DF(d+t)/DF(d)). `refresh()`는 할 일이 없다(Observer).

In [ ]:
d = date(2026, 10, 26)
cf.roll(d)
with evaluation_date(d):
    implied = ql.ImpliedTermStructure(ql.YieldTermStructureHandle(ust), to_ql_date(d))
    print(
        "기준일",
        cf.handle.referenceDate(),
        "| 5Y DF 차이",
        cf.handle.discount(5.0) - implied.discount(5.0),
    )
cf.roll(ASOF)
cf.handle.referenceDate()

## 요약

| 객체 | 입력 | 출력 | 언제 쓰나 |
|---|---|---|---|
| `evaluation_date(d)` | 날짜 | with 블록 | QuantLib 계산을 특정 평가일에서 |
| `HELPERS[종류](…)` | Quote, 만기, 구간, Context | QuantLib 헬퍼 | `build_curve` 안. 새 시세 종류를 넣을 때 함수 하나 추가 |
| `build_curve(…)` | 레시피, 관례, 시세, asof, linked, 이미 만든 커브, 지수 | (커브, {시세 키: Quote}) | `build_market` 안 |
| `CurveFactors(base, asof, tenors)` | 날 커브, 격자 | 시장 객체 (`handle`, `factors`, `roll`, `base`) | `build_market`이 커브마다 하나 |

## 내 메모

- 